# k-Nearest Neighbors (k-NN) Classifier — Student Performance Dataset
Applying k-NN from scratch on `student_performance_cleaned.xlsx`

## Step 1: Upload Your Dataset
Run this cell first to upload `student_performance_cleaned.xlsx` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload student_performance_cleaned.xlsx here

## Step 2: Import Required Libraries

In [ ]:
import math
import random
import pandas as pd
from collections import Counter

## Step 3: Load and Preview the Dataset

In [ ]:
df = pd.read_excel('student_performance_cleaned.xlsx')
print('Shape:', df.shape)
df.head()

## Step 4: Encode Categorical Columns
k-NN uses distance calculations, so all text values must be converted to numbers.

In [ ]:
df['Class_Participation'] = df['Class_Participation'].map({'Low': 0, 'Medium': 1, 'High': 2})
df['Internet_Access']     = df['Internet_Access'].map({'No': 0, 'Yes': 1})
df['Parent_Education']    = df['Parent_Education'].map({'Primary': 0, 'Secondary': 1, 'Graduate': 2})
df['Tutoring_Support']    = df['Tutoring_Support'].map({'No': 0, 'Yes': 1})
df['Stress_Level']        = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
df['Class_Label']         = df['Class_Label'].map({'Fail': 0, 'Pass': 1})

print('Encoding complete!')
df.head()

## Step 5: Prepare Features and Labels

In [ ]:
feature_cols = [
    'Study_Hours_Per_Day', 'Attendance_Rate(%)',
    'Previous_Exam_Score', 'Assignment_Completion(%)',
    'Sleep_Hours', 'Class_Participation',
    'Internet_Access', 'Parent_Education',
    'Tutoring_Support', 'Stress_Level'
]

# Convert to list of ([features], label) — same format as lab notebook
dataset = [
    (list(row[feature_cols]), row['Class_Label'])
    for _, row in df.iterrows()
]

print(f'Total samples: {len(dataset)}')
print(f'Sample entry : {dataset[0]}')

## Step 6: Shuffle and Split (80:20)

In [ ]:
random.seed(42)  # for reproducibility
random.shuffle(dataset)

split_index = int(0.8 * len(dataset))
train_data  = dataset[:split_index]
test_data   = dataset[split_index:]

print(f'Training samples : {len(train_data)}')
print(f'Testing  samples : {len(test_data)}')

## Step 7: Distance Functions

In [ ]:
# ==========================================
# Euclidean Distance
# Formula: sqrt(sum((x1-x2)^2))
# ==========================================
def euclidean_distance(point1, point2):
    total = 0
    for i in range(len(point1)):
        total += (point1[i] - point2[i]) ** 2
    return math.sqrt(total)

# ==========================================
# Manhattan Distance
# Formula: sum(|x1-x2|)
# ==========================================
def manhattan_distance(point1, point2):
    total = 0
    for i in range(len(point1)):
        total += abs(point1[i] - point2[i])
    return total

print('Distance functions defined!')

## Step 8: k-NN Prediction Function

In [ ]:
def knn_predict(train_data, test_point, k=3, metric='euclidean'):
    distances = []

    # Step 1: Calculate distance to every training point
    for features, label in train_data:
        if metric == 'euclidean':
            dist = euclidean_distance(features, test_point)
        else:
            dist = manhattan_distance(features, test_point)
        distances.append((dist, label))

    # Step 2: Sort by distance (ascending)
    distances.sort(key=lambda x: x[0])

    # Step 3: Take k nearest labels
    k_nearest_labels = [label for _, label in distances[:k]]

    # Step 4: Majority vote
    prediction = Counter(k_nearest_labels).most_common(1)[0][0]

    return prediction

print('k-NN prediction function defined!')

## Step 9: Train and Evaluate the Model

In [ ]:
K_VALUE = 3
METRIC  = 'euclidean'   # change to 'manhattan' to try Manhattan distance

correct = 0

print(f'---- k-NN Evaluation (k={K_VALUE}, metric={METRIC}, 80:20 split) ----\n')

for features, actual_label in test_data:
    prediction = knn_predict(train_data, features, k=K_VALUE, metric=METRIC)
    class_name = 'Pass' if prediction == 1 else 'Fail'
    actual_name = 'Pass' if actual_label == 1 else 'Fail'
    match = '✓' if prediction == actual_label else '✗'
    print(f'  Predicted: {class_name:<5}  |  Actual: {actual_name:<5}  |  {match}')
    if prediction == actual_label:
        correct += 1

accuracy = correct / len(test_data)
print(f'\nCorrect: {correct}/{len(test_data)}')
print(f'Accuracy (k={K_VALUE}, {METRIC}): {accuracy * 100:.2f}%')

---
## Task 1: Experiment with K Values, Distance Metrics, and Split Ratios
Runs all combinations automatically and stores results in a table.

In [ ]:
k_values    = [1, 3, 5, 7]
metrics     = ['euclidean', 'manhattan']
split_ratios = [0.8, 0.7, 0.6]   # train size: 80:20, 70:30, 60:40

results = []

for split in split_ratios:
    random.seed(42)
    random.shuffle(dataset)
    idx        = int(split * len(dataset))
    tr         = dataset[:idx]
    te         = dataset[idx:]
    split_name = f'{int(split*100)}:{100 - int(split*100)}'

    for k in k_values:
        for metric in metrics:
            correct = sum(
                1 for features, actual in te
                if knn_predict(tr, features, k=k, metric=metric) == actual
            )
            acc = correct / len(te) * 100
            results.append({
                'Split Ratio' : split_name,
                'K Value'     : k,
                'Distance'    : metric.capitalize(),
                'Train Size'  : len(tr),
                'Test Size'   : len(te),
                'Correct'     : correct,
                'Accuracy (%)'  : round(acc, 2)
            })

results_df = pd.DataFrame(results)
print('=== Task 1 Results Table ===')
print(results_df.to_string(index=False))

### Best Configuration

In [ ]:
best = results_df.loc[results_df['Accuracy (%)'].idxmax()]
print('Best Configuration:')
print(best.to_string())